In [1]:
!pip install librosa

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install librosa numpy pandas scikit-learn tensorflow matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import librosa
print(librosa.__version__)

0.11.0


In [4]:
import os
import numpy as np
import pandas as pd
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [5]:
paths = []
labels = []

dataset_path = "dataset"

for folder in os.listdir(dataset_path):
    actor_folder = os.path.join(dataset_path, folder)

    for file in os.listdir(actor_folder):
        paths.append(os.path.join(actor_folder, file))

        emotion_code = file.split("-")[2]

        if emotion_code == "01":
            labels.append("neutral")
        elif emotion_code == "02":
            labels.append("calm")
        elif emotion_code == "03":
            labels.append("happy")
        elif emotion_code == "04":
            labels.append("sad")
        elif emotion_code == "05":
            labels.append("angry")
        elif emotion_code == "06":
            labels.append("fearful")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset'

In [6]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05', 'Actor_06', 'Actor_07', 'Actor_08', 'Actor_09', 'Actor_10', 'Actor_11', 'Actor_12', 'Actor_13', 'Actor_14', 'Actor_15', 'Actor_16', 'Actor_17', 'Actor_18', 'Actor_19', 'Actor_20', 'Actor_21', 'Actor_22', 'Actor_23', 'Actor_24', 'Emotion_Recognition.ipynb']


In [7]:
paths = []
labels = []

dataset_path = "."

for folder in os.listdir(dataset_path):
    if folder.startswith("Actor_"):
        actor_folder = os.path.join(dataset_path, folder)

        for file in os.listdir(actor_folder):
            paths.append(os.path.join(actor_folder, file))

            emotion_code = file.split("-")[2]

            if emotion_code == "01":
                labels.append("neutral")
            elif emotion_code == "02":
                labels.append("calm")
            elif emotion_code == "03":
                labels.append("happy")
            elif emotion_code == "04":
                labels.append("sad")
            elif emotion_code == "05":
                labels.append("angry")
            elif emotion_code == "06":
                labels.append("fearful")
            elif emotion_code == "07":
                labels.append("disgust")
            elif emotion_code == "08":
                labels.append("surprised")

In [8]:
print(len(paths))
print(len(labels))

1440
1440


In [9]:
df = pd.DataFrame()
df["path"] = paths
df["label"] = labels

df.head()

,path,label
0,.\Actor_01\03-01-01-01-01-01-01.wav,neutral
1,.\Actor_01\03-01-01-01-01-02-01.wav,neutral
2,.\Actor_01\03-01-01-01-02-01-01.wav,neutral
3,.\Actor_01\03-01-01-01-02-02-01.wav,neutral
4,.\Actor_01\03-01-02-01-01-01-01.wav,calm


In [10]:
def extract_features(file_path):
    audio, sample_rate = librosa.load(file_path, duration=3)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=40
    )

    return np.mean(mfcc.T, axis=0)

In [11]:
X = []
y = []

for path, label in zip(paths, labels):
    features = extract_features(path)

    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

In [12]:
print(X.shape)

(1440, 40)


In [13]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

y_categorical = to_categorical(y_encoded)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42
)

In [15]:
model = Sequential()

model.add(Dense(256, activation='relu',
                input_shape=(40,)))

model.add(Dropout(0.3))

model.add(Dense(128, activation='relu'))

model.add(Dropout(0.3))

model.add(Dense(
    y_categorical.shape[1],
    activation='softmax'
))

C:\Users\shaik\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.1354 - loss: 36.4139 - val_accuracy: 0.1458 - val_loss: 3.9918
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1354 - loss: 7.0649 - val_accuracy: 0.1458 - val_loss: 2.0795
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1345 - loss: 2.5061 - val_accuracy: 0.1458 - val_loss: 2.0781
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1311 - loss: 2.2661 - val_accuracy: 0.1458 - val_loss: 2.0768
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1363 - loss: 2.2324 - val_accuracy: 0.1458 - val_loss: 2.0759
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1267 - loss: 2.1276 - val_accuracy: 0.1111 - val_loss: 2.0748
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1319 - loss: 2.1337 - val_accuracy: 0.1111 - val_loss: 2.0740
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1302 - loss: 2.1245 - val_accuracy: 0.1111 - val_lo

In [18]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:", accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1111 - loss: 2.0677
Accuracy: 0.1111111119389534


In [19]:
import os

dataset_path = "."

count = 0

for folder in os.listdir(dataset_path):
    actor_folder = os.path.join(dataset_path, folder)

    if os.path.isdir(actor_folder):
        files = os.listdir(actor_folder)
        count += len(files)

print("Total audio files:", count)

Total audio files: 1441


In [20]:
import pandas as pd

print(pd.Series(labels).value_counts())

calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96
Name: count, dtype: int64


In [21]:
paths = []
labels = []

In [22]:
print(X.shape)
print(y.shape)

(1440, 40)
(1440,)


In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

In [3]:
paths = []
labels = []

dataset_path = "."

emotion_dict = {
    "01":"neutral",
    "02":"calm",
    "03":"happy",
    "04":"sad",
    "05":"angry",
    "06":"fearful",
    "07":"disgust",
    "08":"surprised"
}

for folder in os.listdir(dataset_path):

    actor_folder = os.path.join(dataset_path, folder)

    if os.path.isdir(actor_folder):

        for file in os.listdir(actor_folder):

            if file.endswith(".wav"):

                emotion_code = file.split("-")[2]

                paths.append(
                    os.path.join(actor_folder,file)
                )

                labels.append(
                    emotion_dict[emotion_code]
                )

print("Audio Files:",len(paths))

Audio Files: 1440


In [4]:
print(pd.Series(labels).value_counts())

calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96
Name: count, dtype: int64


In [5]:
def extract_features(file_path):

    audio, sr = librosa.load(
        file_path,
        duration=3,
        offset=0.5
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40
    )

    mfcc = np.mean(
        mfcc.T,
        axis=0
    )

    return mfcc

In [6]:
X = []
y = []

for path,label in zip(paths,labels):

    features = extract_features(path)

    X.append(features)

    y.append(label)

X = np.array(X)

y = np.array(y)

print(X.shape)
print(y.shape)

(1440, 40)
(1440,)


In [7]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

print(X.shape)

(1440, 40)


In [8]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

y_categorical = to_categorical(
    y_encoded
)

print(y_categorical.shape)

(1440, 8)


In [9]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [10]:
model = Sequential()

model.add(
    Dense(
        512,
        activation='relu',
        input_shape=(40,)
    )
)

model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(
    Dense(
        256,
        activation='relu'
    )
)

model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(
    Dense(
        8,
        activation='softmax'
    )
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\shaik\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 512)                 │          20,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 8)                   │           1,032 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 189,832 (741.53 KB)

 Trainable params: 188,040 (734.53 KB)

 Non-trainable params: 1,792 (7.00 KB)

In [11]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(
        X_test,
        y_test
    )
)

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.2526 - loss: 2.3462 - val_accuracy: 0.3438 - val_loss: 1.8288
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.3898 - loss: 1.8002 - val_accuracy: 0.4062 - val_loss: 1.7155
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4219 - loss: 1.6143 - val_accuracy: 0.4097 - val_loss: 1.6503
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4809 - loss: 1.4927 - val_accuracy: 0.3958 - val_loss: 1.5839
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5425 - loss: 1.2689 - val_accuracy: 0.4167 - val_loss: 1.5310
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5703 - loss: 1.2217 - val_accuracy: 0.4653 - val_loss: 1.4359
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5998 - loss: 1.1199 - val_accuracy: 0.4896 - val_loss: 1.3671
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6024 - loss: 1.0939 - val_accuracy: 0.

In [12]:
loss,accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:",accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7188 - loss: 1.0945 
Accuracy: 0.71875


In [13]:
model.save(
    "emotion_model.h5"
)

In [14]:
with open(
    "label_encoder.pkl",
    "wb"
) as f:

    pickle.dump(
        encoder,
        f
    )

In [16]:
with open(
    "scaler.pkl",
    "wb"
) as f:

    pickle.dump(
        scaler,
        f
    )